# AA Simulations — mol-dynamics (Rust) with OpenMM fallback

Runs all-atom MD for 6 molecules, extracts CG distributions (with dihedrals),
and parameterizes the force field. Downloads `cg_forcefield.json` at the end.

In [ ]:
# Install all dependencies upfront (including OpenMM as fallback)
!pip install -q mol-dynamics MDAnalysis biopython scikit-learn scipy numpy openmm
!pip install -q git+https://github.com/WestonVoglesonger/adaptive-cg.git

In [ ]:
# Check which engines are available
USE_RUST = False
USE_OPENMM = False

try:
    import mol_dynamics
    print(f'mol-dynamics loaded')
    USE_RUST = True
except ImportError:
    print('mol-dynamics not available')

try:
    import openmm
    print(f'OpenMM loaded')
    USE_OPENMM = True
except ImportError:
    print('OpenMM not available')

if not USE_RUST and not USE_OPENMM:
    raise RuntimeError('Neither mol-dynamics nor OpenMM available!')

print(f'\nPrimary engine: {"Rust (mol-dynamics)" if USE_RUST else "OpenMM"}')

In [ ]:
import os, urllib.request
from pathlib import Path
import numpy as np

MOLECULES = ['1CRN', '1UBQ', '1LYZ', '1BNA', '1IGD', '1PGA']

# Create directories
os.makedirs('data/structures', exist_ok=True)
os.makedirs('data/trajectories', exist_ok=True)

# Download PDB and CIF files
for mol in MOLECULES:
    pdb_path = f'data/structures/{mol}.pdb'
    cif_path = f'data/structures/{mol}.cif'
    if not os.path.exists(pdb_path):
        url = f'https://files.rcsb.org/download/{mol}.pdb'
        urllib.request.urlretrieve(url, pdb_path)
        print(f'Downloaded {pdb_path}')
    if not os.path.exists(cif_path):
        url = f'https://files.rcsb.org/download/{mol}.cif'
        urllib.request.urlretrieve(url, cif_path)
        print(f'Downloaded {cif_path}')

print('All structures ready.')

In [ ]:
# === Rust engine: mol-dynamics ===

def run_aa_rust(mol_id, n_steps=5000, dt=0.002):
    """Run AA MD using mol-dynamics (Rust+CUDA)."""
    from mol_dynamics import (
        MdState, MdConfig, MolDynamics, FfParamSet, FfMolType, MmCif,
        prepare_peptide_mmcif, Integrator,
    )

    cif_path = f'data/structures/{mol_id}.cif'
    out_dir = Path(f'data/trajectories/{mol_id}')
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f'Loading {cif_path}...')
    protein = MmCif.load(cif_path)
    param_set = FfParamSet.new_amber()
    prepare_peptide_mmcif(protein, param_set.peptide_ff_q_map, 7.0)

    mol = MolDynamics(
        ff_mol_type=FfMolType.Peptide,
        atoms=protein.atoms,
        atom_posits=None,
        bonds=[],
        adjacency_list=None,
        static_=False,
        mol_specific_params=None,
        atom_init_velocities=None,
        bonded_only=False,
    )

    cfg = MdConfig()
    cfg.integrator = Integrator.LangevinMiddle
    cfg.temp_target = 300.0

    md = MdState(cfg, [mol], param_set)

    print(f'Running {n_steps} steps (dt={dt} ps)...')
    for i in range(n_steps):
        md.step(dt)
        if (i + 1) % 500 == 0:
            snap = md.snapshots[-1]
            print(f'  Step {i+1}/{n_steps}: KE={snap.energy_kinetic:.1f}, PE={snap.energy_potential:.1f}')

    # Save as DCD + PDB via MDAnalysis for compatibility with acg extract
    import MDAnalysis as mda

    positions = []
    for idx, snap in enumerate(md.snapshots):
        if idx % 10 == 0:
            pos = np.array([[p.x, p.y, p.z] for p in snap.atom_posits])
            positions.append(pos)
    positions = np.array(positions)
    print(f'Collected {len(positions)} frames, {positions.shape[1]} atoms')

    pdb_path = f'data/structures/{mol_id}.pdb'
    u = mda.Universe(pdb_path)
    solvated_pdb = out_dir / f'{mol_id}_solvated.pdb'

    with mda.Writer(str(solvated_pdb), u.atoms.n_atoms) as w:
        w.write(u.atoms)

    with mda.Writer(str(out_dir / f'{mol_id}_traj.dcd'), u.atoms.n_atoms) as w:
        for pos in positions:
            u.atoms.positions = pos[:u.atoms.n_atoms] * 10  # nm -> Angstrom
            w.write(u.atoms)

    print(f'Saved to {out_dir}')
    return True


# === OpenMM fallback ===

def run_aa_openmm(mol_id, n_steps=500000, dt=0.002):
    """Run AA MD using OpenMM."""
    from adaptive_cg.core.simulation import run_aa_simulation, SimulationConfig

    pdb_path = Path(f'data/structures/{mol_id}.pdb')
    out_dir = Path(f'data/trajectories/{mol_id}')
    out_dir.mkdir(parents=True, exist_ok=True)

    config = SimulationConfig(
        temperature=300.0,
        pressure=1.0,
        timestep_fs=dt * 1000,
        n_steps=n_steps,
        equilibration_steps=50000,
        save_interval=2000,
    )

    run_aa_simulation(pdb_path, out_dir, config, verbose=True)
    return True

In [ ]:
# Run simulations for all molecules
results = {}

for mol_id in MOLECULES:
    traj_dir = Path(f'data/trajectories/{mol_id}')
    if list(traj_dir.glob('*_traj.dcd')):
        print(f'{mol_id}: already exists, skipping')
        results[mol_id] = True
        continue

    print(f'\n{"="*50}')
    print(f'Simulating {mol_id}')
    print(f'{"="*50}')

    # Try Rust first, then OpenMM
    ok = False
    if USE_RUST:
        try:
            ok = run_aa_rust(mol_id)
        except Exception as e:
            print(f'Rust failed: {e}')

    if not ok and USE_OPENMM:
        print('Using OpenMM...')
        try:
            ok = run_aa_openmm(mol_id)
        except Exception as e:
            print(f'OpenMM failed: {e}')

    results[mol_id] = ok

print(f'\nResults:')
for mol_id, ok in results.items():
    print(f'  {mol_id}: {"OK" if ok else "FAILED"}')

In [ ]:
# Extract CG distributions (with dihedrals) for all molecules
!pip install -q --force-reinstall git+https://github.com/WestonVoglesonger/adaptive-cg.git

for mol_id in MOLECULES:
    traj_dir = Path(f'data/trajectories/{mol_id}')
    if not list(traj_dir.glob('*_traj.dcd')):
        print(f'Skipping {mol_id} (no trajectory)')
        continue
    print(f'\nExtracting {mol_id}...')
    !python -m adaptive_cg extract {mol_id}

In [ ]:
# Parameterize force field (with dihedrals)
!python -m adaptive_cg parameterize

In [ ]:
# Download the force field
from google.colab import files
files.download('data/forcefield/cg_forcefield.json')
print('Done! Copy cg_forcefield.json to your project: data/forcefield/')